## Air Quality Index (AQI) — IQAir to Neo4j

This notebook fetches real-time air quality and weather data from the IQAir API for cities across the Pyrenees corridor — the French regions of Occitanie and Nouvelle-Aquitaine, and the Spanish regions of Aragon, Catalonia and Navarre — and loads the results into Neo4j.

The Neo4j data model is:

```cypher
(:City {name, country, state, lat, lon})
  -[:HAS_READING]->
(:Reading {timestamp, aqi_us, aqi_category, main_pollutant,
           temperature, humidity, wind_speed})

(:City)-[:NEIGHBORS {distance_km}]-(:City)
```

City nodes are stable across runs. Each run appends new Reading nodes, so the database accumulates a time series. A `NEIGHBORS` relationship is created between City nodes within 150 km of each other.

### Before running the notebook

Export the following in your shell:

```bash
export NEO4J_URI=bolt://127.0.0.1:7687
export NEO4J_USERNAME=your_username_here
export NEO4J_PASSWORD=your_password_here
export NEO4J_DATABASE=your_database_name_here
export IQAIR_API_KEY=your_iqair_api_key_here
```

### Install and Import Packages

In [1]:
%pip install neo4j==5.28.4 \
             requests==2.34.2 \
             tabulate==0.10.0 \
             tqdm==4.70.0 --quiet

print("Install complete.")

Note: you may need to restart the kernel to use updated packages.
Install complete.


In [2]:
import math
import os
import requests
import time

from datetime import datetime, timezone
from neo4j import GraphDatabase
from tabulate import tabulate
from tqdm.notebook import tqdm

### Configuration

In [3]:
NEO4J_URI      = os.environ["NEO4J_URI"]
NEO4J_USERNAME = os.environ["NEO4J_USERNAME"]
NEO4J_PASSWORD = os.environ["NEO4J_PASSWORD"]
NEO4J_DATABASE = os.environ["NEO4J_DATABASE"]
IQAIR_API_KEY  = os.environ["IQAIR_API_KEY"]

print("Credentials set.")

Credentials set.


In [4]:
# IQAir free tier: 5 requests per minute, so 1 request every 12 seconds
API_DELAY_SECONDS = 12

# Cities within this distance (km) will be linked as NEIGHBORS
NEIGHBOR_DISTANCE_KM = 200

# Regions to query — French and Spanish sides of the Pyrenees corridor
REGIONS = [
    {"country": "France", "states": ["Occitanie", "Nouvelle-Aquitaine"]},
    {"country": "Spain",  "states": ["Aragon", "Catalonia", "Navarre"]},
]

### Helper Functions

In [5]:
def haversine_km(lat1, lon1, lat2, lon2):
    """Calculate the great-circle distance in km between two points."""
    R = 6371
    d_lat = math.radians(lat2 - lat1)
    d_lon = math.radians(lon2 - lon1)
    a = (math.sin(d_lat / 2) ** 2 +
         math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) *
         math.sin(d_lon / 2) ** 2)
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

def get_aqi_category(aqi):
    """Map a US AQI value to its standard category label."""
    if aqi <= 50:  return "Good"
    if aqi <= 100: return "Moderate"
    if aqi <= 150: return "Unhealthy for Sensitive Groups"
    if aqi <= 200: return "Unhealthy"
    if aqi <= 300: return "Very Unhealthy"
    return "Hazardous"

def fetch_cities(country, state, api_key):
    """Return a list of city name strings for the given state and country."""
    url = f"http://api.airvisual.com/v2/cities"
    params = {"state": state, "country": country, "key": api_key}
    resp = requests.get(url, params = params)
    data = resp.json()
    if data["status"] == "success":
        return [c["city"] for c in data["data"]]
    print(f"  Warning: could not fetch cities for {state}, {country}: {data.get('data')}")
    return []

def fetch_city_data(country, state, city, api_key):
    """Fetch current weather and pollution data for a single city.
    Returns a dict of all fields, or None on failure.
    """
    url = f"http://api.airvisual.com/v2/city"
    params = {"city": city, "state": state, "country": country, "key": api_key}
    resp = requests.get(url, params = params)
    data = resp.json()

    if data["status"] != "success":
        print(f"  Warning: no data for {city}: {data.get('data')}")
        return None

    loc       = data["data"]["location"]["coordinates"]
    weather   = data["data"]["current"]["weather"]
    pollution = data["data"]["current"]["pollution"]

    return {
        "country":        country,
        "state":          state,
        "city":           city,
        "lat":            loc[1],
        "lon":            loc[0],
        "timestamp":      datetime.now(timezone.utc).isoformat(),
        "aqi_us":         pollution.get("aqius"),
        "aqi_cn":         pollution.get("aqicn"),
        "aqi_category":   get_aqi_category(pollution.get("aqius", 0)),
        "main_pollutant": pollution.get("mainus"),
        "temperature":    weather.get("tp"),
        "pressure":       weather.get("pr"),
        "humidity":       weather.get("hu"),
        "wind_speed":     weather.get("ws"),
        "wind_direction": weather.get("wd"),
    }

### Neo4j Helper Functions

In [6]:
def create_constraints(session):
    """Create uniqueness constraints to prevent duplicate City nodes."""
    session.run("""
        CREATE CONSTRAINT city_unique IF NOT EXISTS
        FOR (c:City) REQUIRE (c.name, c.country) IS UNIQUE
    """)

def upsert_city(session, record):
    """Merge a City node (create if not exists, update coordinates)."""
    session.run("""
        MERGE (c:City {name: $city, country: $country})
        SET c.state = $state,
            c.lat   = $lat,
            c.lon   = $lon
    """, **record)

def create_reading(session, record):
    """Create a new Reading node and link it to the City."""
    session.run("""
        MATCH (c:City {name: $city, country: $country})
        CREATE (r:Reading {
            timestamp:      $timestamp,
            aqi_us:         $aqi_us,
            aqi_cn:         $aqi_cn,
            aqi_category:   $aqi_category,
            main_pollutant: $main_pollutant,
            temperature:    $temperature,
            pressure:       $pressure,
            humidity:       $humidity,
            wind_speed:     $wind_speed,
            wind_direction: $wind_direction
        })
        CREATE (c)-[:HAS_READING]->(r)
    """, **record)

### Quick Test: City Count by Region

Before running the full pipeline, we fetch only the city lists to check coverage. This uses one API call per state (5 calls total at the free tier rate limit). If any region returns fewer cities than expected, we can adjust the regions before committing to the full run.

In [7]:
# Quick city count test — no AQI data fetched yet
test_locations = []

for region in REGIONS:
    country = region["country"]
    print(f"\n{country}:")
    for state in region["states"]:
        cities = fetch_cities(country, state, IQAIR_API_KEY)
        test_locations.extend([(country, state, c) for c in cities])
        print(f"  {state}: {len(cities)} cities")
        time.sleep(API_DELAY_SECONDS)

print(f"\nTotal cities available: {len(test_locations)}")
print("\nIf coverage looks thin, adjust REGIONS before proceeding.")


France:
  Occitanie: 22 cities
  Nouvelle-Aquitaine: 19 cities

Spain:
  Aragon: 3 cities
  Catalonia: 39 cities
  Navarre: 9 cities

Total cities available: 92

If coverage looks thin, adjust REGIONS before proceeding.


### Step 1: Fetch City List from IQAir

In [8]:
locations = []

for region in REGIONS:
    country = region["country"]
    for state in tqdm(region["states"], desc=f"{country}"):
        cities = fetch_cities(country, state, IQAIR_API_KEY)
        for city in cities:
            locations.append((country, state, city))
        print(f"  {state}: {len(cities)} cities")
        time.sleep(API_DELAY_SECONDS)

print(f"\nTotal: {len(locations)} cities")

France:   0%|          | 0/2 [00:00<?, ?it/s]

  Occitanie: 22 cities
  Nouvelle-Aquitaine: 19 cities


Spain:   0%|          | 0/3 [00:00<?, ?it/s]

  Aragon: 3 cities
  Catalonia: 39 cities
  Navarre: 9 cities

Total: 92 cities


### Step 2: Fetch AQI and Weather Data per City

In [9]:
records = []
failed  = []

for country, state, city in tqdm(locations, desc="Fetching AQI data"):
    record = fetch_city_data(country, state, city, IQAIR_API_KEY)
    if record:
        records.append(record)
    else:
        failed.append(f"{city}, {country}")
    time.sleep(API_DELAY_SECONDS)

print(f"Fetched: {len(records)} cities.")
if failed:
    print(f"Failed:  {len(failed)}: {', '.join(failed)}")

Fetching AQI data:   0%|          | 0/92 [00:00<?, ?it/s]

Fetched: 91 cities.
Failed:  1: Polinya, Spain


### Step 3: Load Data into Neo4j

In [10]:
driver = GraphDatabase.driver(
    NEO4J_URI,
    auth = (NEO4J_USERNAME, NEO4J_PASSWORD)
)

print(driver.verify_connectivity())  # None is expected
print("Connection created.")

None
Connection created.


In [11]:
with driver.session(database = NEO4J_DATABASE) as session:

    create_constraints(session)

    for record in tqdm(records, desc="Loading cities"):
        upsert_city(session, record)
        create_reading(session, record)

    print(f"City nodes and Reading nodes loaded: {len(records)}")

    neighbor_count = 0
    pairs = [(a, b) for i, a in enumerate(records) for b in records[i + 1:]]
    for a, b in tqdm(pairs, desc="Creating NEIGHBORS"):
        dist = haversine_km(a["lat"], a["lon"], b["lat"], b["lon"])
        if dist <= NEIGHBOR_DISTANCE_KM:
            session.run("""
                MATCH (a:City {name: $city_a, country: $country_a})
                MATCH (b:City {name: $city_b, country: $country_b})
                MERGE (a)-[r:NEIGHBORS]-(b)
                SET r.distance_km = $distance_km
            """,
                city_a      = a["city"],  country_a   = a["country"],
                city_b      = b["city"],  country_b   = b["country"],
                distance_km = round(dist, 1)
            )
            neighbor_count += 1

    print(f"NEIGHBORS relationships created: {neighbor_count}")

print("\nNeo4j load complete.")

Loading cities:   0%|          | 0/91 [00:00<?, ?it/s]

City nodes and Reading nodes loaded: 91


Creating NEIGHBORS:   0%|          | 0/4095 [00:00<?, ?it/s]

NEIGHBORS relationships created: 1426

Neo4j load complete.


### Step 4: Verify the Data in Neo4j

In [12]:
with driver.session(database = NEO4J_DATABASE) as session:
    result = session.run("MATCH (c:City) RETURN count(c) AS cities")
    print(f"City nodes:              {result.single()['cities']}")
    result = session.run("MATCH (r:Reading) RETURN count(r) AS readings")
    print(f"Reading nodes:           {result.single()['readings']}")
    result = session.run("MATCH ()-[n:NEIGHBORS]-() RETURN count(n) AS neighbors")
    print(f"NEIGHBORS relationships: {result.single()['neighbors']}")

    print("\nLatest AQI reading per city:")
    result = session.run("""
        MATCH (c:City)-[:HAS_READING]->(r:Reading)
        WITH c, r ORDER BY r.timestamp DESC
        WITH c, collect(r)[0] AS latest
        RETURN c.name AS city, c.country AS country,
               latest.aqi_us AS aqi_us,
               latest.aqi_category AS category
        ORDER BY latest.aqi_us DESC
    """)
    rows = [[row["city"], row["country"], row["aqi_us"], row["category"]]
            for row in result]
    print(tabulate(
        rows,
        headers  = ["City", "Country", "AQI US", "Category"],
        tablefmt = "simple"
    ))

City nodes:              102
Reading nodes:           369
NEIGHBORS relationships: 3462

Latest AQI reading per city:
City                                   Country      AQI US  Category
-------------------------------------  ---------  --------  ------------------------------
Beauzelle                              France          157  Unhealthy
Toulouse                               France          123  Unhealthy for Sensitive Groups
Poitiers                               France          108  Unhealthy for Sensitive Groups
Villiers-en-Bois                       France          105  Unhealthy for Sensitive Groups
Niort                                  France          105  Unhealthy for Sensitive Groups
Limoges                                France          105  Unhealthy for Sensitive Groups
Angoulême                              France          105  Unhealthy for Sensitive Groups
Gueret                                 France          104  Unhealthy for Sensitive Groups
Périgueux      

### Step 5: Teardown

In [13]:
driver.close()
print("Connections closed.")

Connections closed.
